# Lab 1 — Semantic Search in ES|QL: How Vector Search Actually Works

**Thesis:** You can run state-of-the-art semantic search — finding documents by *meaning*, not just matching keywords — without writing a single line of embedding code. And in this track you express it in **ES|QL**, Elasticsearch's piped query language: one readable statement, run in **Kibana Discover** or from Python.

## What you'll learn
- How `semantic_text` fields work and why you write zero embedding code
- The ES|QL `MATCH` function — and how it does semantic search automatically on a `semantic_text` field
- The 4-step query mechanism: ES → Elastic Inference Service (EIS) → Jina v5 → ANN
- Why `METADATA _score` is the switch that turns a filter into a ranked search

## Two ways to run every query in this lab
- **Kibana Discover (ES|QL mode)** — paste the `FROM ... | ...` blocks into the query bar. Results render as a table. *(See the Discover orientation in the lab page.)*
- **This notebook** — the same queries through `es.esql.query()`, so you can script and post-process them.

## Before you start
- **In Instruqt:** `ES_ENDPOINT` and `ES_API_KEY` are pre-configured — just run the cells.
- **Re-running from the repo:** `export ES_ENDPOINT=https://...` and `export ES_API_KEY=...`

## New to vectors? Start here (2-minute primer)

If this is your first time with semantic search, here's the whole idea before we touch any code.

**An embedding is text turned into a list of numbers that captures *meaning*.** The model reads a piece of text and outputs a vector — a few hundred numbers — positioned so that text about *similar things* lands close together, and text about *different things* lands far apart. No math required on your end; the model does it.

**That's what makes semantic search different from keyword search.** Keyword search (BM25) matches the *words* you typed. Semantic search matches the *meaning*. So a query like **"securing cluster traffic"** can find a document about **"TLS encryption"** — even though they share no words — because both vectors land in the same neighborhood.

**`semantic_text` is the field type that does all of this for you.** When you map a field as `semantic_text`, Elasticsearch automatically calls the embedding model — at *index time* for every document, and again at *query time* for every search. You write **zero embedding code**.

**Where ES|QL comes in:** in ES|QL you search a field with the `MATCH` function. Point `MATCH` at a normal `text` field and you get BM25 keyword search; point it at a `semantic_text` field and Elasticsearch runs a *semantic vector search* instead — same function, the field type decides. That's the whole trick you'll use all workshop.

In [ ]:
# --- Workshop helpers (inline — same block across all ES|QL notebooks) ---
# ES|QL edition: every search runs through es.esql.query() instead of es.search().
# Defined inline so this notebook is self-contained and runs from the repo too.

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

# request_timeout=120: RERANK and COMPLETION (Labs 4-5) call inference per row and
# can take several seconds — the default 10s would time out the LLM step.
es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=120)

def esql(query, **params):
    """Run an ES|QL query with named parameters (?name in the query string).

    Usage:  esql(QUERY, q="securing cluster traffic")
    ES|QL named params take the form params=[{"name": value}, ...]. If your pinned
    client rejects named params, switch to positional `?` and params=[value, ...] —
    never f-string the query text in (injection + teaches the wrong pattern).
    """
    param_list = [{k: v} for k, v in params.items()] if params else None
    return es.esql.query(query=query, params=param_list, format="json")

def rows(resp):
    """Turn an ES|QL response ({columns, values}) into a list of dicts keyed by column."""
    cols = [c["name"] for c in resp["columns"]]
    return [dict(zip(cols, vals)) for vals in resp["values"]]

def show_esql(resp, fields=("id", "title", "summary"), score=True):
    """Pretty-print ES|QL rows as a ranked table (mirrors the DSL notebooks' show_hits)."""
    data = rows(resp)
    if not data:
        print("  (no rows)"); return
    for rank, r in enumerate(data, 1):
        cols = "  ".join(str(r.get(f, "")) for f in fields)
        sc = r.get("_score")
        s = f"  {sc:.4f}" if score and sc is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

print("✓ ES|QL helpers loaded")

In [ ]:
# Sanity check: confirm we're connected and the corpus is indexed.
# In ES|QL, COUNT(*) over the index is the equivalent of es.count().
info = es.info()
print(f"Connected to Elasticsearch {info['version']['number']}")

resp = esql("FROM aiewf-workshop-docs | STATS docs = COUNT(*)")
count = rows(resp)[0]["docs"]
print(f"Index '{INDEX}': {count} documents indexed")
if count == 0:
    print("\n⚠ No documents found — ingest may still be running. Wait 30s and retry.")

## What does a `semantic_text` field look like in practice?

The corpus has a field called `body_semantic` mapped as `semantic_text`. ES|QL itself has no `_mapping` command, so we inspect the field two ways:
- In **Discover**, the left-hand field list shows every field and its type.
- Here in the notebook, we use the regular client (`es.indices.get_mapping`) — ES|QL is for *querying*, the admin APIs are still there when you need them.

Pay attention to:
- `type: semantic_text` — the field type that enables all of this
- `inference_id` — **auto-assigned** (`.jina-embeddings-v5-text-small`). You didn't set it; Elastic Serverless provisioned it for you.

In [ ]:
# Fetch the index mapping and focus on the semantic_text field.
mapping = es.indices.get_mapping(index=INDEX)
props = mapping[INDEX]["mappings"]["properties"]

print("=== body_semantic field mapping ===")
print(json.dumps(props.get("body_semantic", {}), indent=2))

print("\n=== all field types ===")
for field, defn in props.items():
    print(f"  {field:<20} {defn.get('type', 'object')}")

In [ ]:
# The mapping pointed us at the embedding endpoint: .jina-embeddings-v5-text-small
# Fetch its config to see the model, dimensions, similarity, and chunking settings.
EMBEDDING_ENDPOINT = ".jina-embeddings-v5-text-small"
ep = es.inference.get(inference_id=EMBEDDING_ENDPOINT)
print(json.dumps(ep.body, indent=2))

## Your first semantic query in ES|QL — note: zero embedding code

Here's the query, as you'd type it in Discover:

```esql
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(body_semantic, "securing cluster traffic")
| SORT _score DESC
| LIMIT 5
| KEEP id, title, summary, _score
```

Read it top to bottom like a pipeline:
- `FROM aiewf-workshop-docs METADATA _score` — read the index, and **ask for the relevance score**. Without `METADATA _score`, `MATCH` acts as a pure yes/no filter with no ranking. This is the single most important ES|QL-search habit.
- `WHERE MATCH(body_semantic, "...")` — `body_semantic` is a `semantic_text` field, so this runs a **semantic vector search**, not keyword matching.
- `SORT _score DESC | LIMIT 5` — rank by relevance, keep the top 5.
- `KEEP ...` — choose the columns to return (the ES|QL equivalent of `_source`).

The top document is about **TLS / transport-layer encryption** — semantically *about* securing cluster traffic, even though the body never contains that phrase. No client-side embedding, no vector math.

In [ ]:
QUERY = """
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(body_semantic, ?q)
| SORT _score DESC
| LIMIT 5
| KEEP id, title, summary, _score
"""

q = "securing cluster traffic"
print(f"Query: {q!r}\n")
show_esql(esql(QUERY, q=q))

## What just happened? The 4-step mechanism

```
Your query string
      │
      ▼
  Elasticsearch  ──── sends query text ───►  Elastic Inference Service (EIS)
      │                                              │
      │                                    Jina v5 embedding model
      │                                              │
      │          ◄── 1024-dim float vector ──────────┘
      │
      ▼
  HNSW ANN index  (approximate nearest-neighbour search over stored doc vectors)
      │
      ▼
  Top-K semantically similar documents
```

**ANN / HNSW** — Elasticsearch uses Hierarchical Navigable Small World graphs for approximate nearest-neighbour search. "Approximate" trades a tiny accuracy margin for being **orders of magnitude faster** than brute-force cosine similarity. At this corpus size it's effectively exact; at 10M vectors it's still fast.

**Why this matters:** at index time, when the ingest script wrote `body_semantic: doc["body"]`, ES sent the text to EIS, got a vector back, and stored it in the HNSW index. At query time the same thing happens to your query string. The single word `MATCH` in your ES|QL kicked off all of it.

In [ ]:
# More "wow" queries — each finds semantically related docs with little keyword overlap.
for q in ["how do I back up my cluster data", "users can't connect to Kibana"]:
    print(f"\n{'='*60}\nQUERY: {q!r}")
    show_esql(esql(QUERY, q=q))

## How chunking works — and what `semantic_text` hides from you

ANN search ranks *chunks*, not whole documents. If a 10,000-word document has one relevant paragraph buried in section 7, you want that paragraph to surface — not the whole document averaged together.

**The default `semantic_text` chunking strategy (`sentence`):**
- Up to ~250 words per chunk, boundaries always at sentence endings
- 1 sentence of overlap between adjacent chunks for continuity

**At query time** your query vector is compared against *all* chunks across *all* documents, and the **max similarity across any chunk** becomes the document's score — so a 5-chunk doc competes fairly with a 1-chunk doc.

**Matryoshka dimensions** — Jina v5 is trained so the first N dimensions already carry most of the meaning, so you can truncate stored vectors (1024 → 512 → 256) to cut storage with minimal recall loss. Configured in the mapping at index time; mentioned here as a "when you hit scale" lever, not something to change today.

## Summary — and a question

You just ran semantic search in ES|QL that:
- Finds documents by **concept**, not keyword overlap
- Uses a **production-grade embedding model** (Jina v5, 1024 dims) managed by Elastic
- Handles **chunking, embedding, and ANN indexing** automatically
- Is **one readable pipeline** — `FROM | WHERE MATCH | SORT | LIMIT | KEEP`

---

**The setup question for Lab 2:**

> If semantic search is this good at understanding meaning, why would you ever need old-fashioned keyword (BM25) search?

Try this — in Discover or here — and look hard at the ranking:

```esql
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(body_semantic, "exit code 137")
| SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score
```

Does the result look right? What doc *should* be #1? Lab 2 shows you exactly why semantic search struggles here — and why you need both.

---
*Continue in Discover → Lab 2 assignment, or open `lab2-esql-where-vector-breaks.ipynb`*